In [1]:

%pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

%pip install pandas requests beautifulsoup4 matplotlib numpy scipy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

import json
import re
import sqlite3
from pathlib import Path

import pandas as pd


# --------------------------------------------------
# 1. PROJECT PATHS
# --------------------------------------------------

RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/clean")
DB_PATH = Path("data/allbirds.db")

BASE_URL = "https://www.allbirds.com/products/"


# --------------------------------------------------
# 2. CLEAN PRICE VALUES
# --------------------------------------------------

def clean_price(value):
    """Convert a price to a number, or None if invalid."""

    if value is None or str(value).strip() == "":
        return None

    try:
        cleaned = re.sub(r"[^0-9.\-]", "", str(value))

        if not cleaned:
            return None

        return float(cleaned)

    except (ValueError, TypeError):
        return None


# --------------------------------------------------
# 3. INITIALISE DATABASE TABLES
# --------------------------------------------------

def initialise_database(conn):
    """Create tables without deleting existing records."""

    conn.execute("PRAGMA foreign_keys = ON")

    conn.executescript("""
        CREATE TABLE IF NOT EXISTS products (
            product_id INTEGER PRIMARY KEY,
            title TEXT,
            product_type TEXT,
            vendor TEXT,
            tags TEXT,
            handle TEXT,
            product_url TEXT
        );

        CREATE TABLE IF NOT EXISTS observations (
            observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
            product_id INTEGER NOT NULL,
            scraped_at TEXT NOT NULL,
            variant_id INTEGER NOT NULL,
            variant_title TEXT,
            size TEXT,
            price REAL,
            available INTEGER,
            FOREIGN KEY (product_id)
                REFERENCES products(product_id),
            UNIQUE (product_id, variant_id, scraped_at)
        );
    """)


# --------------------------------------------------
# 4. IDENTIFY PRODUCT SIZE
# --------------------------------------------------

def identify_size(product, variant):
    """Find the size from the product's option fields."""

    options = product.get("options") or []
    values = []

    for index, option in enumerate(options, start=1):
        value = variant.get(f"option{index}")

        if value is not None:
            values.append((
                str(option.get("name", "")),
                str(value)
            ))

    for name, value in values:
        if "size" in name.lower():
            return value

    # Use the variant title if no size option was identified.
    title = str(variant.get("title", "")).strip()

    if title.lower() not in ("", "default title", "default"):
        return title

    return None


# --------------------------------------------------
# 5. LOAD AND CLEAN RAW JSON FILES
# --------------------------------------------------

def main():
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    files = sorted(RAW_DIR.glob("allbirds_raw_*.json"))

    if not files:
        raise SystemExit(
            "No raw JSON files found. Run scraper.py first."
        )

    products_seen = {}
    observations = []

    for file in files:
        print("Reading:", file.name)

        try:
            snapshot = json.loads(
                file.read_text(encoding="utf-8")
            )
        except (json.JSONDecodeError, OSError) as error:
            print("Skipping invalid file:", file.name, error)
            continue

        scraped_at = snapshot.get("scraped_at")

        if not scraped_at:
            print("Skipping snapshot without timestamp:", file.name)
            continue

        for product in snapshot.get("products", []):
            if not isinstance(product, dict):
                continue

            pid = product.get("id")

            if pid is None:
                continue

            # Keep product IDs consistent with SQLite INTEGER keys.
            try:
                pid = int(pid)
            except (ValueError, TypeError):
                print("Skipping non-numeric product ID:", pid)
                continue

            handle = product.get("handle")
            tags = product.get("tags", [])

            if isinstance(tags, list):
                tags = ", ".join(map(str, tags))
            elif tags is None:
                tags = ""

            products_seen[pid] = {
                "product_id": pid,
                "title": product.get("title"),
                "product_type": product.get("product_type"),
                "vendor": product.get("vendor"),
                "tags": tags,
                "handle": handle,
                "product_url": (
                    f"{BASE_URL}{handle}" if handle else None
                )
            }

            variants = product.get("variants") or []

            for variant in variants:
                if not isinstance(variant, dict):
                    continue

                variant_id = variant.get("id")

                # The observation key requires a variant ID.
                if variant_id is None:
                    continue

                try:
                    variant_id = int(variant_id)
                except (ValueError, TypeError):
                    print("Skipping invalid variant ID:", variant_id)
                    continue

                available_value = variant.get("available")

                if available_value is None:
                    available = None
                elif isinstance(available_value, bool):
                    available = int(available_value)
                elif str(available_value).strip().lower() in (
                    "true", "1", "yes"
                ):
                    available = 1
                elif str(available_value).strip().lower() in (
                    "false", "0", "no"
                ):
                    available = 0
                else:
                    available = None

                observations.append({
                    "product_id": pid,
                    "scraped_at": str(scraped_at),
                    "variant_id": variant_id,
                    "variant_title": variant.get("title"),
                    "size": identify_size(product, variant),
                    "price": clean_price(variant.get("price")),
                    "available": available
                })

    # --------------------------------------------------
    # 6. CREATE DATAFRAMES
    # --------------------------------------------------

    product_columns = [
        "product_id", "title", "product_type", "vendor",
        "tags", "handle", "product_url"
    ]

    observation_columns = [
        "product_id", "scraped_at", "variant_id",
        "variant_title", "size", "price", "available"
    ]

    product_df = pd.DataFrame(
        list(products_seen.values()),
        columns=product_columns
    )

    obs_df = pd.DataFrame(
        observations,
        columns=observation_columns
    )

    # Remove duplicate observations within the cleaned dataset.
    if not obs_df.empty:
        obs_df = obs_df.drop_duplicates(
            subset=["product_id", "variant_id", "scraped_at"],
            keep="last"
        )

    # --------------------------------------------------
    # 7. EXPORT CLEANED CSV FILES
    # --------------------------------------------------

    product_df.to_csv(
        CLEAN_DIR / "products_cleaned.csv",
        index=False,
        encoding="utf-8-sig"
    )

    obs_df.to_csv(
        CLEAN_DIR / "observations_cleaned.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------
    # 8. STORE DATA IN SQLITE
    # --------------------------------------------------

    with sqlite3.connect(DB_PATH) as conn:
        initialise_database(conn)

        # Insert or update product details without replacing the table.
        conn.executemany("""
            INSERT INTO products (
                product_id, title, product_type, vendor,
                tags, handle, product_url
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(product_id) DO UPDATE SET
                title = excluded.title,
                product_type = excluded.product_type,
                vendor = excluded.vendor,
                tags = excluded.tags,
                handle = excluded.handle,
                product_url = excluded.product_url
        """, product_df.itertuples(index=False, name=None))

        # Insert each observation only once for its unique key.
        # Existing daily observations are preserved on reruns.
        conn.executemany("""
            INSERT INTO observations (
                product_id, scraped_at, variant_id,
                variant_title, size, price, available
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(product_id, variant_id, scraped_at)
            DO NOTHING
        """, obs_df.itertuples(index=False, name=None))

        product_count = conn.execute(
            "SELECT COUNT(*) FROM products"
        ).fetchone()[0]

        observation_count = conn.execute(
            "SELECT COUNT(*) FROM observations"
        ).fetchone()[0]

        distinct_days = conn.execute("""
            SELECT COUNT(DISTINCT substr(scraped_at, 1, 10))
            FROM observations
        """).fetchone()[0]

    # --------------------------------------------------
    # 9. DISPLAY RESULTS
    # --------------------------------------------------

    print("\nDAY 5: DATA CLEANING AND DATABASE LOADING")
    print("-----------------------------------------")
    print("Raw files processed:", len(files))
    print("Unique products in current files:", len(product_df))
    print("Unique observations in current files:", len(obs_df))
    print("Products stored in database:", product_count)
    print("Observations stored in database:", observation_count)
    print("Distinct observation dates:", distinct_days)
    print("Cleaned CSV files:", CLEAN_DIR.resolve())
    print("SQLite database:", DB_PATH.resolve())


if __name__ == "__main__":
    main()


Reading: allbirds_raw_2026-10-15.json
Reading: allbirds_raw_2026-10-16.json
Reading: allbirds_raw_2026-10-16_142513.json
Reading: allbirds_raw_2026-10-17.json
Reading: allbirds_raw_2026-10-17_200657.json
Reading: allbirds_raw_2026-10-18.json
Reading: allbirds_raw_2026-10-19.json

DAY 5: DATA CLEANING AND DATABASE LOADING
-----------------------------------------
Raw files processed: 7
Unique products in current files: 692
Unique observations in current files: 52178
Products stored in database: 692
Observations stored in database: 52178
Distinct observation dates: 5
Cleaned CSV files: D:\Desktop\WEB SCRAPING\data\clean
SQLite database: D:\Desktop\WEB SCRAPING\data\allbirds.db


In [4]:
from pathlib import Path

db_path = Path("data/allbirds.db")

print("Database exists:", db_path.exists())
print("Full path:", db_path.resolve())

Database exists: True
Full path: D:\Desktop\WEB SCRAPING\data\allbirds.db


In [5]:
import sqlite3

db_path = r"D:\Desktop\WEB SCRAPING\data\allbirds.db"

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
    """)

    tables = cursor.fetchall()

    print("Tables in database:")
    for table in tables:
        print(table[0])

Tables in database:
products
observations
sqlite_sequence


In [6]:
import sqlite3

db_path = r"D:\Desktop\WEB SCRAPING\data\allbirds.db"

with sqlite3.connect(db_path) as conn:
    for table in ["products", "observations"]:
        try:
            count = conn.execute(
                f"SELECT COUNT(*) FROM {table}"
            ).fetchone()[0]

            print(f"{table}: {count} records")

        except sqlite3.OperationalError:
            print(f"{table}: table not found")

products: 692 records
observations: 52178 records


In [7]:

import sqlite3
import pandas as pd

db_path = r"D:\Desktop\WEB SCRAPING\data\allbirds.db"

with sqlite3.connect(db_path) as conn:
    products = pd.read_sql_query(
        "SELECT * FROM products", conn
    )

    observations = pd.read_sql_query(
        "SELECT * FROM observations", conn
    )

print("=== DATASET SUMMARY ===")
print("Unique products:", products["product_id"].nunique())
print("Observation records:", len(observations))

print("\n=== COLLECTION DATES ===")
print(
    observations["scraped_at"]
    .dropna()
    .astype(str)
    .str[:10]
    .value_counts()
    .sort_index()
)

print("\n=== MISSING VALUES IN PRODUCTS ===")
print(products.isnull().sum())

print("\n=== MISSING VALUES IN OBSERVATIONS ===")
print(observations.isnull().sum())

print("\n=== PRICE SUMMARY ===")
print(observations["price"].describe())

print("\n=== AVAILABILITY COUNTS ===")
print(observations["available"].value_counts(dropna=False))

print("\n=== PRODUCT TYPES ===")
print(products["product_type"].value_counts(dropna=False).head(15))


=== DATASET SUMMARY ===
Unique products: 692
Observation records: 52178

=== COLLECTION DATES ===
scraped_at
2026-10-15     7454
2026-10-16    14908
2026-10-17    14908
2026-10-18     7454
2026-10-19     7454
Name: count, dtype: int64

=== MISSING VALUES IN PRODUCTS ===
product_id      0
title           0
product_type    0
vendor          0
tags            0
handle          0
product_url     0
dtype: int64

=== MISSING VALUES IN OBSERVATIONS ===
observation_id    0
product_id        0
scraped_at        0
variant_id        0
variant_title     0
size              0
price             0
available         0
dtype: int64

=== PRICE SUMMARY ===
count    52178.000000
mean       101.844579
std         34.325367
min          0.800000
25%         75.000000
50%        105.000000
75%        130.000000
max        165.000000
Name: price, dtype: float64

=== AVAILABILITY COUNTS ===
available
0    35772
1    16406
Name: count, dtype: int64

=== PRODUCT TYPES ===
product_type
Shoes                      